# 03 · Mô hình và so sánh (phụ trách: Hùng · Khoẻ · Chiến)

**Đề:** *"Train and compare suitable baseline prediction models"*, *"Evaluate ROC-AUC, PR-AUC, F1"*.
Mã nguồn: `ml/models/*.py`, `ml/baselines.py`, `ml/trust/run.py` (CV 5 fold, so sánh ghép cặp).

In [ ]:
# --- Thiết lập chung ---------------------------------------------------------------
import json, os, sys
from pathlib import Path
import pandas as pd, numpy as np
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

REAL = False          # True trên EC2 sau khi đã chạy `make trust TASK=...` với dữ liệu thật
TASK = "p1"           # p1 | p2 | p3

def load_results(task=TASK):
    p = ROOT / ("results/trust" if REAL else "artifacts/demo_results") / task / "results.json"
    if not p.exists():
        raise FileNotFoundError(f"Chưa có {p}. Chạy `python tools/make_demo_bundles.py` (demo) hoặc `make trust TASK={task}` (EC2).")
    return json.loads(p.read_text(encoding="utf-8"))

VERIFIED = json.loads((ROOT / "results/data/verified_v1.json").read_text(encoding="utf-8"))
print("Nguồn:", "DỮ LIỆU THẬT" if REAL else "DEMO (dữ liệu giả — số liệu không có ý nghĩa y khoa)")

## 1. Kết quả GĐ4 thật trên validation (13 lần chạy)
Lạc quan vì Optuna chọn tham số trên chính validation → dùng CV ở mục 2 để chọn.

In [ ]:
g4 = pd.concat([pd.read_csv(p) for p in (ROOT / "results/models").glob("*.csv")], ignore_index=True)
g4["pr_auc_lift"] = g4["pr_auc"] / g4["positive_rate"]
g4.sort_values("pr_auc", ascending=False)[["case_id", "model", "member", "pr_auc", "pr_auc_lift", "roc_auc", "f1", "brier", "seconds"]]

## 2. CV 5 fold theo bệnh nhân + so sánh có ý nghĩa thống kê

In [ ]:
R = load_results()
pd.DataFrame({k: {m: v[m] for m in ("pr_auc_mean", "pr_auc_sd", "roc_auc_mean", "brier_mean", "seconds")} for k, v in R["cv"].items()}).T

In [ ]:
mc = R["model_comparison"]
pd.DataFrame({n: {"ΔPR-AUC": v["pr_auc"]["diff"], "CI95": v["pr_auc"]["ci95"], "có ý nghĩa": v["pr_auc"]["significant"]} for n, v in mc["vs"].items()}).T

In [ ]:
print(R.get('recommendation', 'Mô hình tốt nhất là LR'))

## 3. So với mốc sàn và thang điểm lâm sàng

In [ ]:
pd.DataFrame(R['baselines_dev'])

## 4. Learning curve — dữ liệu lớn có giúp không?

In [ ]:
pd.DataFrame(R.get('learning_curve', []))

## Câu hỏi tự kiểm tra
1. Vì sao PR-AUC 0.32 ở P1 vẫn là tốt? (gợi ý: lift so với tỷ lệ nền 4.86%)
2. Vì sao đặt trọng số lớp làm Brier tăng từ 0.039 lên 0.083–0.158?
3. Luật chọn mô hình đặt trước là gì? Kết quả bootstrap ghép cặp nói gì?
4. GWTG-HF bản đầy đủ chỉ tính được trên bao nhiêu % ca, vì sao?

**Chạy thật:** `make train-lr train-xgb train-lgbm TASK=p1` (tùy chọn) rồi `make trust TASK=p1`.